In [ ]:
import numpy as np
import imageio

import os

from natsort import natsorted

def _binarize(a):

    """Convert to {0,1}. Works for 0/255, booleans, etc."""

    return (a > 0).astype(np.uint8)

def confusion_counts(gt, pred):

    """Return TP, FP, FN, TN for binary masks."""

    gt  = _binarize(gt).ravel()

    pred = _binarize(pred).ravel()

    TP = np.sum((pred == 1) & (gt == 1))

    FP = np.sum((pred == 1) & (gt == 0))

    FN = np.sum((pred == 0) & (gt == 1))

    TN = np.sum((pred == 0) & (gt == 0))

    return TP, FP, FN, TN

def precision_recall_f1(gt, pred):

    """Precision, Recall (TPR), F1 (== Dice for binary)."""

    TP, FP, FN, _ = confusion_counts(gt, pred)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0

    f1        = (2 * precision * recall / (precision + recall)

                 if (precision + recall) > 0 else 0.0)

    return precision, recall, f1

def dice(gt, pred):

    """Dice Similarity Coefficient (== F1 for binary)."""

    TP, FP, FN, _ = confusion_counts(gt, pred)

    return (2 * TP) / (2 * TP + FP + FN) if (2*TP + FP + FN) > 0 else 0.0

def iou(gt, pred):

    """Jaccard / Intersection-over-Union."""

    TP, FP, FN, _ = confusion_counts(gt, pred)

    return TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0.0



def load_tif_stack(folder_path):

    file_list = natsorted([f for f in os.listdir(folder_path) if f.endswith('.tif')])

    stack = [imageio.imread(os.path.join(folder_path, f)) for f in file_list]

    return np.stack(stack, axis=0)

pred = load_tif_stack("/home/a5549/Project_roothair/Filtered masks/Sav_S6/Sav_S6_python_results")

gt = load_tif_stack("/home/a5549/Project_roothair/Filtered masks/Sav_S6/Sav_S6_VG_results")

prec, rec, f1 = precision_recall_f1(gt, pred)

dsc = dice(gt, pred)       # equal to f1 for binary

jacc = iou(gt, pred)

print(f"Precision: {prec:.4f}")

print(f"Recall (TPR): {rec:.4f}")

print(f"F1 (Dice): {f1:.4f}")

print(f"IoU: {jacc:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Voxel-wise segmentation performance evaluation
# ---------------------------------------------------------------------------
# This script computes standard binary segmentation metrics by comparing a
# pipeline-generated segmentation volume against a reference (ground-truth)
# segmentation on a voxel-by-voxel basis. Both volumes are loaded as 3D
# stacks from folders of individual 2D TIFF slices, binarised, and then
# evaluated using four complementary metrics: Precision, Recall (TPR),
# Dice Similarity Coefficient (DSC), and Intersection over Union (IoU).
# Together these metrics characterise both the overlap quality and the
# balance between false positives and false negatives, providing a complete
# picture of segmentation performance for a single scan session.
# ---------------------------------------------------------------------------

import numpy as np
import imageio
import os
from natsort import natsorted

# ---------------------------------------------------------------------------
# Configuration — replace these values before running
# ---------------------------------------------------------------------------
# Absolute path to the folder containing the pipeline-generated segmentation
# slices (one 2D TIFF per slice, grayscale, any non-zero value = foreground).
pred_folder = "/path/to/predicted/segmentation"

# Absolute path to the folder containing the reference (ground-truth)
# segmentation slices (same format and slice count as pred_folder).
gt_folder = "/path/to/reference/segmentation"

# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------
def _binarize(a):
    """Convert an array to a binary mask with values in {0, 1}.
    Compatible with uint8 masks (0/255), boolean arrays, and labelled masks
    where any non-zero value represents foreground."""
    return (a > 0).astype(np.uint8)

def confusion_counts(gt, pred):
    """Compute the four entries of the binary confusion matrix.

    Both inputs are binarised and flattened before comparison so that the
    function is agnostic to array shape and input dtype.

    Returns
    -------
    TP : int  — true positives  (predicted foreground, ground-truth foreground)
    FP : int  — false positives (predicted foreground, ground-truth background)
    FN : int  — false negatives (predicted background, ground-truth foreground)
    TN : int  — true negatives  (predicted background, ground-truth background)
    """
    gt   = _binarize(gt).ravel()
    pred = _binarize(pred).ravel()
    TP = int(np.sum((pred == 1) & (gt == 1)))
    FP = int(np.sum((pred == 1) & (gt == 0)))
    FN = int(np.sum((pred == 0) & (gt == 1)))
    TN = int(np.sum((pred == 0) & (gt == 0)))
    return TP, FP, FN, TN

def precision_recall_f1(gt, pred):
    """Compute Precision, Recall (TPR), and F1 score for binary masks.

    Precision = TP / (TP + FP)  — of all predicted positives, how many are correct.
    Recall    = TP / (TP + FN)  — of all actual positives, how many were found.
    F1        = 2 * (Precision * Recall) / (Precision + Recall)
              = DSC for binary masks.

    Returns 0.0 for any metric whose denominator is zero (i.e. degenerate cases
    where the predicted or ground-truth mask is entirely empty).
    """
    TP, FP, FN, _ = confusion_counts(gt, pred)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

def dice(gt, pred):
    """Dice Similarity Coefficient (DSC) — equivalent to F1 for binary masks.

    DSC = 2*TP / (2*TP + FP + FN)

    Range: [0, 1], where 1 = perfect overlap. Returns 0.0 if both masks are
    empty (degenerate case).
    """
    TP, FP, FN, _ = confusion_counts(gt, pred)
    return (2 * TP) / (2 * TP + FP + FN) if (2 * TP + FP + FN) > 0 else 0.0

def iou(gt, pred):
    """Intersection over Union (IoU) — also known as the Jaccard index.

    IoU = TP / (TP + FP + FN)

    Range: [0, 1], where 1 = perfect overlap. More sensitive than DSC to
    false positives and false negatives because the union grows with both
    error types, penalising them more strongly. Returns 0.0 if the union
    is empty.
    """
    TP, FP, FN, _ = confusion_counts(gt, pred)
    return TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0.0

def load_tif_stack(folder_path):
    """Load a folder of 2D TIFF slices into a 3D NumPy array (Z, Y, X).

    natsorted ensures slices are loaded in natural numerical order
    (e.g. slice_2.tif before slice_10.tif) rather than lexicographic order.
    """
    file_list = natsorted([f for f in os.listdir(folder_path) if f.endswith('.tif')])
    stack = [imageio.imread(os.path.join(folder_path, f)) for f in file_list]
    return np.stack(stack, axis=0)

# ---------------------------------------------------------------------------
# Load segmentation volumes
# ---------------------------------------------------------------------------
pred = load_tif_stack(pred_folder)
gt   = load_tif_stack(gt_folder)

# ---------------------------------------------------------------------------
# Compute and report metrics
# ---------------------------------------------------------------------------
prec, rec, f1 = precision_recall_f1(gt, pred)
dsc  = dice(gt, pred)    # Mathematically equal to f1 for binary masks
jacc = iou(gt, pred)

print(f"Precision    : {prec:.4f}")
print(f"Recall (TPR) : {rec:.4f}")
print(f"F1 / DSC     : {f1:.4f}")
print(f"IoU (Jaccard): {jacc:.4f}")